# Central Texas — Comprehensive Watershed Report

**Rivers:** Colorado River at Austin · Brazos River at Waco · Guadalupe River at Victoria

Three major river systems draining Central Texas, each with decades of continuous USGS
streamflow monitoring. This report examines flow regime, long-term trends, climate
teleconnections (ENSO/PDO), flood/drought patterns, and seasonal behavior across all
three basins.

In [ ]:
# ── imports & paths ──────────────────────────────────────────────────────
import sys
from pathlib import Path

REPO = Path(".").resolve().parent
sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from src import flow_metrics as fm
from tools.nwis_gauge import GaugeProvider
from tools.climate_index import ClimateIndexProvider
from tools.monthly_flow import MONTH_ABBR

NAS_ROOT = "/Volumes/home/data/hydro-art"

%matplotlib inline
plt.rcParams.update({
    "figure.facecolor": "#07080c",
    "axes.facecolor": "#10121b",
    "axes.edgecolor": "#232838",
    "axes.labelcolor": "#e6ebf5",
    "text.color": "#e6ebf5",
    "xtick.color": "#8891a8",
    "ytick.color": "#8891a8",
    "figure.dpi": 140,
    "savefig.dpi": 140,
    "font.size": 10,
})

# neon palette
CYAN = "#00ffff"
MAGENTA = "#ff4d9a"
LIME = "#00ff9c"
VIOLET = "#9d00ff"
AMBER = "#ff9c3a"
RED = "#ff4d5e"
COOL_BLUE = "#00e5ff"
MUTED = "#8891a8"
LIGHT = "#e6ebf5"

def complete_years(obs):
    """Filter to years where all 12 months are finite (no NaN gaps)."""
    return {y: v for y, v in obs.items() if np.all(np.isfinite(v))}

print("hydro-art Central Texas report — ready")

## 1. Data Acquisition

In [ ]:
# ── gauge data ────────────────────────────────────────────────────────────
colorado = GaugeProvider("08158000", NAS_ROOT)
colorado_obs = colorado.monthly_means(1960, 2024)
print(f"Colorado River: {colorado.name}")
print(f"  {len(colorado_obs)} years: {min(colorado_obs)}–{max(colorado_obs)}")

brazos = GaugeProvider("08096500", NAS_ROOT)
brazos_obs = brazos.monthly_means(1960, 2024)
print(f"\nBrazos River: {brazos.name}")
print(f"  {len(brazos_obs)} years: {min(brazos_obs)}–{max(brazos_obs)}")

guadalupe = GaugeProvider("08176500", NAS_ROOT)
guadalupe_obs = guadalupe.monthly_means(1940, 2024)
print(f"\nGuadalupe River: {guadalupe.name}")
print(f"  {len(guadalupe_obs)} years: {min(guadalupe_obs)}–{max(guadalupe_obs)}")

# ── climate indices ────────────────────────────────────────────────────────
oni_prov = ClimateIndexProvider("oni", NAS_ROOT)
pdo_prov = ClimateIndexProvider("pdo", NAS_ROOT)

oni = oni_prov.index_by_year(1940, 2024)
pdo = pdo_prov.index_by_year(1940, 2024)

print(f"\nONI (ENSO): {len(oni)} years")
print(f"PDO: {len(pdo)} years")

# Complete-year filtered copies
colorado_c = complete_years(colorado_obs)
brazos_c = complete_years(brazos_obs)
guadalupe_c = complete_years(guadalupe_obs)
print(f"\nComplete years — Colorado: {len(colorado_c)}, Brazos: {len(brazos_c)}, Guadalupe: {len(guadalupe_c)}")

# Handy lists for iteration
RIVERS = [
    ("Colorado River at Austin", colorado_c, CYAN),
    ("Brazos River at Waco", brazos_c, MAGENTA),
    ("Guadalupe River at Victoria", guadalupe_c, LIME),
]

In [ ]:
# ── quick data summary table ──────────────────────────────────────────────
for name, obs, _ in RIVERS:
    years = sorted(obs.keys())
    annual = [np.mean(v) for v in obs.values()]
    peak = [np.max(v) for v in obs.values()]
    low = [np.min(v) for v in obs.values()]
    print(f"\n{name} ({years[0]}–{years[-1]}, {len(years)} yr):")
    print(f"  Mean annual flow:  {np.mean(annual):10,.1f} cfs")
    print(f"  Median annual:     {np.median(annual):10,.1f} cfs")
    print(f"  Peak monthly mean: {np.max(peak):10,.1f} cfs")
    print(f"  Lowest monthly:    {np.min(low):10,.1f} cfs")
    print(f"  CV (annual):       {np.std(annual)/np.mean(annual):10.3f}")

---
## 2. Monthly Hydrograph Spaghetti

In [ ]:
def plot_hydrographs(obs, title, ax):
    years = sorted(obs.keys())
    cmap = plt.get_cmap("viridis")
    span = max(years) - min(years) or 1
    for y in years:
        c = cmap((y - min(years)) / span)
        ax.plot(range(1, 13), obs[y], color=c, lw=0.8, alpha=0.7)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)
    sm = plt.cm.ScalarMappable(cmap="viridis",
        norm=plt.Normalize(vmin=min(years), vmax=max(years)))
    ax.figure.colorbar(sm, ax=ax, label="year", pad=0.01)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, _) in zip(axes, RIVERS):
    plot_hydrographs(obs, f"{name}\n({min(obs)}–{max(obs)})", ax)
plt.tight_layout(); plt.show()

**What this shows:** Every year's monthly flow drawn as a single line (Jan–Dec), color-coded from oldest (purple) to newest (yellow). Each line is one year's "shape" — where that year's water arrived across the calendar.

**How to read it:** Wide vertical spread = high year-to-year variability for that month. If newer (yellow) lines cluster differently than older (purple) ones, the seasonal pattern is shifting over time. Texas rivers are highly variable — expect a thick, overlapping bundle rather than the tidy Pacific Northwest fan. Spikes in spring/fall reflect convective storm seasons.

---
## 3. Typical Year (Envelope Plot)

In [ ]:
def plot_typical_year(obs, title, ax):
    years = sorted(obs.keys())
    stack = np.array([obs[y] for y in years])
    mean = np.nanmean(stack, axis=0)
    p10 = np.nanpercentile(stack, 10, axis=0)
    p25 = np.nanpercentile(stack, 25, axis=0)
    p75 = np.nanpercentile(stack, 75, axis=0)
    p90 = np.nanpercentile(stack, 90, axis=0)
    m = range(1, 13)
    ax.fill_between(m, p10, p90, color=CYAN, alpha=0.08, label="P10–P90")
    ax.fill_between(m, p25, p75, color=CYAN, alpha=0.15, label="P25–P75")
    ax.plot(m, mean, color=CYAN, lw=2, label="mean")
    ax.plot(m, np.nanmedian(stack, axis=0), color=AMBER, lw=1.5, ls="--", label="median")
    cot = fm.center_of_timing(mean)
    ax.axvline(cot, color=AMBER, ls=":", lw=1, label=f"COT {cot:.1f}")
    ax.set_xticks(list(m))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=7, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, _) in zip(axes, RIVERS):
    plot_typical_year(obs, f"{name} — Typical Year", ax)
plt.tight_layout(); plt.show()

**What this shows:** The "average shape" of a water year — the mean (solid cyan) and median (dashed amber) monthly flow, wrapped in probability envelopes (inner band = 25th–75th percentile, outer = 10th–90th). The vertical dotted line marks the **Center of Timing (COT)** — the month by which half the year's total flow has passed.

**How to read it:** The wider the shaded band, the more unpredictable that month's flow is. When mean sits well above median, the distribution is right-skewed — a few extreme flood years pull the average up. COT near month 5–6 is typical for rain-dominated Texas rivers; a shift toward earlier months would indicate changing storm seasonality.

---
## 4. Long-Term Annual Trend

In [ ]:
def plot_long_record(obs, title, ax, color=CYAN):
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    mk = fm.mann_kendall(annual)
    slope = fm.sens_slope(annual)
    x = np.arange(len(years))
    fit = np.median(annual) + slope * (x - np.median(x))
    ax.plot(years, annual, "o-", color=color, lw=1.2, ms=3, label="annual mean")
    ax.plot(years, fit, "--", color=VIOLET, lw=1.5,
            label=f"Sen slope {slope:+.1f} cfs/yr")
    # 10-year rolling mean
    if len(annual) >= 10:
        kernel = np.ones(10) / 10
        rolling = np.convolve(annual, kernel, mode="valid")
        ax.plot(years[4:4+len(rolling)], rolling, color=AMBER, lw=1.8,
                alpha=0.85, label="10-yr rolling mean")
    ax.set_ylabel("annual-mean flow (cfs)")
    ax.set_title(f"{title}\nMann-Kendall: {mk.trend} (τ={mk.tau:+.3f}, p={mk.p:.4f})")
    ax.legend(fontsize=7, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, color) in zip(axes, RIVERS):
    plot_long_record(obs, f"{name} — Long Record", ax, color)
plt.tight_layout(); plt.show()

**What this shows:** Each dot is one year's average flow (mean of 12 monthly values). The purple dashed line is the **Sen's slope** — a robust, outlier-resistant linear trend. The amber curve is a 10-year rolling mean that reveals decadal wet/dry cycles. The subtitle reports the **Mann-Kendall test**: τ measures trend strength (−1 to +1), p < 0.05 means statistically significant.

**How to read it:** If the Sen line tilts up, flows are increasing over the record; tilting down means declining. The rolling mean shows whether changes are gradual or driven by distinct wet/dry epochs (common in Texas — the 1950s drought and 2010s floods are often visible as deep troughs and sharp peaks). "No trend" with low τ and high p means the record is dominated by variability, not a directional shift.

---
## 5. Summer Low-Flow Trend

In [ ]:
def plot_low_flow(obs, title, ax, color=LIME):
    years = sorted(obs.keys())
    # Texas summer: Jun–Sep (indices 5–8)
    low = np.array([float(np.nanmin(obs[y][5:9])) for y in years])
    slope = fm.sens_slope(low)
    mk = fm.mann_kendall(low)
    x = np.arange(len(years))
    fit = np.median(low) + slope * (x - np.median(x))
    ax.plot(years, low, "o-", color=color, lw=1.2, ms=3, label="summer low (Jun–Sep)")
    ax.plot(years, fit, "--", color=VIOLET, lw=1.5,
            label=f"Sen {slope:+.2f} cfs/yr")
    ax.set_ylabel("summer-low flow (cfs)")
    ax.set_title(f"{title}\ntrend: {mk.trend} (p={mk.p:.4f})")
    ax.legend(fontsize=7, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, color) in zip(axes, RIVERS):
    plot_low_flow(obs, f"{name} — Low Flow", ax, color)
plt.tight_layout(); plt.show()

**What this shows:** The lowest monthly-mean flow during each summer (June–September) plotted year by year with a Sen's slope trend line. Summer low flow is the critical ecological and water-supply metric — it determines whether a river can sustain aquatic life, dilute effluent, and supply municipal withdrawals during peak demand.

**How to read it:** Declining low-flow trend = growing drought stress. Years near zero are severe — the river approached intermittent conditions. In Central Texas, dams and reservoirs (especially on the Colorado at Austin) regulate summer baseflow, so trends here reflect both climate and reservoir management decisions.

---
## 6. Peak Flow Trend

In [ ]:
def plot_peak_flow(obs, title, ax, color=RED):
    years = sorted(obs.keys())
    peaks = np.array([float(np.nanmax(obs[y])) for y in years])
    slope = fm.sens_slope(peaks)
    mk = fm.mann_kendall(peaks)
    x = np.arange(len(years))
    fit = np.median(peaks) + slope * (x - np.median(x))
    ax.plot(years, peaks, "o-", color=color, lw=1.2, ms=3, label="annual peak month")
    ax.plot(years, fit, "--", color=VIOLET, lw=1.5,
            label=f"Sen {slope:+.1f} cfs/yr")
    ax.set_ylabel("peak monthly-mean flow (cfs)")
    ax.set_title(f"{title}\ntrend: {mk.trend} (p={mk.p:.4f})")
    ax.legend(fontsize=7, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, _) in zip(axes, RIVERS):
    plot_peak_flow(obs, f"{name} — Peak Flow", ax)
plt.tight_layout(); plt.show()

**What this shows:** Each year's highest monthly-mean flow, tracing the flood signal over decades. This captures the magnitude of the wettest month each year — an indicator of flash-flood intensity and storm-season severity.

**How to read it:** Upward trend = storms are getting bigger (or more frequent in a single month). Isolated spikes far above the trend line are landmark flood events. In Central Texas, these often correspond to tropical remnants, stalled fronts, or training thunderstorms. A flat or declining trend would suggest flood regulation (dams) is dampening peak flows.

---
## 7. Flow Duration Curves by Decade

In [ ]:
def plot_decade_fdc(obs, title, ax):
    quantiles = (0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99)
    decades = fm.decade_flow_duration(obs, quantiles)
    cmap = plt.get_cmap("plasma")
    n = len(decades) or 1
    for i, d in enumerate(decades):
        c = cmap(i / n)
        ax.semilogy([q * 100 for q in d.quantiles], d.flows,
                     "o-", color=c, lw=1.5, ms=4, label=f"{d.decade}s")
    ax.set_xlabel("exceedance percentile (%)")
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=7, frameon=False, ncol=2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, _) in zip(axes, RIVERS):
    plot_decade_fdc(obs, f"{name} — Flow Duration by Decade", ax)
plt.tight_layout(); plt.show()

**What this shows:** A flow duration curve (FDC) answers "what flow is exceeded X% of the time?" — plotted on a log scale, one curve per decade. Decades shifting upward = wetter; shifting downward = drier.

**How to read it:** The left side (low exceedance %) represents extreme high flows; the right side represents low/base flows. If the right-hand tails separate across decades, baseflow conditions are changing. If the left-hand tails diverge, flood magnitudes are shifting. Parallel curves mean the whole flow regime scaled proportionally; crossing curves mean wet extremes and dry extremes moved in different directions.

---
## 8. Seasonal Ratio Over Time

In [ ]:
def plot_seasonal_ratio(obs, title, ax, color=CYAN):
    years = sorted(obs.keys())
    # Wet season for TX: Oct–Apr (indices 9,10,11,0,1,2,3)
    ratios = np.array([fm.seasonal_ratio(obs[y], wet=(10, 11, 12, 1, 2, 3, 4)) for y in years])
    slope = fm.sens_slope(ratios)
    mk = fm.mann_kendall(ratios)
    x = np.arange(len(years))
    fit = np.median(ratios) + slope * (x - np.median(x))
    ax.plot(years, ratios, "o-", color=color, lw=1.2, ms=3, label="wet/total")
    ax.plot(years, fit, "--", color=VIOLET, lw=1.5, label=f"Sen {slope:+.4f}/yr")
    ax.axhline(0.5, color=MUTED, ls=":", lw=0.8, alpha=0.5)
    ax.set_ylabel("wet-season fraction")
    ax.set_title(f"{title}\ntrend: {mk.trend} (p={mk.p:.4f})")
    ax.legend(fontsize=7, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, color) in zip(axes, RIVERS):
    plot_seasonal_ratio(obs, f"{name} — Seasonal Ratio", ax, color)
plt.tight_layout(); plt.show()

**What this shows:** The fraction of each year's total flow that arrives during the wet season (October–April for Central Texas). Values above 0.5 mean the wet season dominates; below 0.5 means summer storms contribute more than half.

**How to read it:** A rising trend means flow is concentrating more into the wet season (drier summers, wetter winters). A declining trend means the opposite — summer gaining a larger share. High year-to-year scatter is normal for Texas, where a single tropical event can deliver a month's worth of water in days.

---
## 9. Flashiness Index

In [ ]:
def plot_flashiness(obs, title, ax, color=AMBER):
    years = sorted(obs.keys())
    fi = np.array([fm.flashiness(obs[y]) for y in years])
    slope = fm.sens_slope(fi)
    mk = fm.mann_kendall(fi)
    x = np.arange(len(years))
    fit = np.median(fi) + slope * (x - np.median(x))
    ax.plot(years, fi, "o-", color=color, lw=1.2, ms=3, label="flashiness")
    ax.plot(years, fit, "--", color=VIOLET, lw=1.5, label=f"Sen {slope:+.5f}/yr")
    ax.set_ylabel("flashiness index")
    ax.set_title(f"{title}\ntrend: {mk.trend} (p={mk.p:.4f})")
    ax.legend(fontsize=7, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, _) in zip(axes, RIVERS):
    plot_flashiness(obs, f"{name} — Flashiness", ax)
plt.tight_layout(); plt.show()

**What this shows:** The **Richards-Baker Flashiness Index** — how "spiky" the hydrograph is each year. It measures the ratio of month-to-month flow changes to total flow. Higher values = more abrupt swings between wet and dry months; lower = smoother, more predictable flow.

**How to read it:** Texas rivers are naturally flashy compared to groundwater-fed or snowmelt systems. Rising flashiness over time could indicate intensifying storm patterns, land-use changes (more impervious surface = faster runoff), or declining baseflow. Reservoir-regulated rivers (like the Colorado at Austin) often show lower flashiness than unregulated streams.

---
## 10. Center-of-Timing Drift

In [ ]:
def plot_timing_drift(obs, title, ax, color=CYAN):
    years = sorted(obs.keys())
    cot = np.array([fm.center_of_timing(obs[y]) for y in years])
    slope = fm.sens_slope(cot)
    mk = fm.mann_kendall(cot)
    x = np.arange(len(years))
    fit = np.median(cot) + slope * (x - np.median(x))
    ax.plot(years, cot, "o-", color=color, lw=1.2, ms=3, label="COT (month)")
    ax.plot(years, fit, "--", color=VIOLET, lw=1.5,
            label=f"Sen {slope:+.4f} mo/yr")
    ax.set_yticks(range(1, 13))
    ax.set_yticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("center of timing")
    ax.set_title(f"{title}\ntrend: {mk.trend} (p={mk.p:.4f})")
    ax.legend(fontsize=7, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, color) in zip(axes, RIVERS):
    plot_timing_drift(obs, f"{name} — Timing Drift", ax, color)
plt.tight_layout(); plt.show()

**What this shows:** The **Center of Timing (COT)** for each year — the fractional month by which 50% of the year's total flow has passed. It answers: "Is water arriving earlier or later in the year than it used to?"

**How to read it:** In snowmelt-driven basins, COT drift is a primary climate signal (earlier = warmer winters). In rain-dominated Texas basins, COT is noisier — a single massive spring storm can pull it earlier, a late-year hurricane can push it later. A persistent trend, though, would signal a real shift in storm-season timing. High year-to-year scatter is expected here.

---
## 11. ENSO Composites (El Niño vs. La Niña)

In [ ]:
def plot_composites(obs, index_by_year, title, ax, *, index_name="ONI"):
    comp = fm.composite_hydrographs(obs, index_by_year, warm_min=0.5, cool_max=-0.5)
    m = range(1, 13)
    ax.plot(m, comp.warm, color=RED, lw=2, label=f"warm {index_name} (n={len(comp.warm_years)})")
    ax.plot(m, comp.cool, color=COOL_BLUE, lw=2, label=f"cool {index_name} (n={len(comp.cool_years)})")
    ax.plot(m, comp.neutral, color=MUTED, lw=1.5, ls="--",
            label=f"neutral (n={len(comp.neutral_years)})")
    ax.set_xticks(list(m))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=7, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, _) in zip(axes, RIVERS):
    plot_composites(obs, oni, f"{name} — ENSO Composite", ax)
plt.tight_layout(); plt.show()

**What this shows:** The average monthly hydrograph during **El Niño** (warm ONI > +0.5, red), **La Niña** (cool ONI < −0.5, blue), and **neutral** years (gray dashed). This is the classic ENSO teleconnection analysis — does Pacific Ocean temperature predict Texas river flows?

**How to read it:** In Texas, El Niño winters are typically wetter — if the red line sits above the blue line in Oct–Mar, the ENSO signal is working as expected. Separation between the curves shows teleconnection strength. If all three lines overlap, ENSO has little influence on that basin. The "(n=...)" in the legend tells you how many years went into each composite — small n = less reliable.

---
## 12. Anomaly Stripes

In [ ]:
def plot_anomaly_stripes(obs, title, ax, color_pos=CYAN, color_neg=RED):
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    mean = np.mean(annual)
    anomalies = (annual - mean) / mean * 100  # percent departure
    colors = [color_pos if a >= 0 else color_neg for a in anomalies]
    ax.bar(years, anomalies, color=colors, width=0.8, alpha=0.85)
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_ylabel("% departure from mean")
    ax.set_title(title)

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
for ax, (name, obs, _) in zip(axes, RIVERS):
    plot_anomaly_stripes(obs, f"{name} — Annual Anomaly", ax)
plt.tight_layout(); plt.show()

**What this shows:** Each bar is one year's flow as a percent departure from the long-term mean. Cyan bars = wetter than average; red bars = drier than average. Stacked vertically so you can compare drought/flood synchrony across all three basins.

**How to read it:** Look for multi-year runs of the same color — these are persistent drought or wet epochs. When all three rivers show the same pattern simultaneously, a large-scale climate driver (like ENSO or a persistent ridge/trough) is at work. If one basin diverges (e.g., the Guadalupe floods while the Colorado stays dry), local convective storms or reservoir operations are dominating.

---
## 13. Monthly Heatmap

In [ ]:
def plot_heatmap(obs, title):
    years = sorted(obs.keys())
    grid = np.array([obs[y] for y in years])
    fig, ax = plt.subplots(figsize=(10, max(4, len(years) * 0.12)))
    im = ax.imshow(grid, aspect="auto", cmap="YlGnBu",
                   interpolation="nearest",
                   extent=[0.5, 12.5, years[-1] + 0.5, years[0] - 0.5])
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=8)
    ax.set_ylabel("year")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="flow (cfs)", pad=0.01)
    plt.tight_layout(); plt.show()

for name, obs, _ in RIVERS:
    plot_heatmap(obs, f"{name} — Monthly Flow Heatmap")

**What this shows:** A year × month grid where color intensity represents flow magnitude. Each row is a year; each column is a month. Dark blue = high flow; pale yellow = low flow. This is the most information-dense view in the report — every data point is visible.

**How to read it:** Vertical streaks of dark blue = a specific month that's consistently wet (e.g., spring flood season). Horizontal streaks = an entire year that was wet or dry. Isolated bright spots = individual flood events. Dark bands across all months = mega-wet years (like 2015 in Central Texas). Pale horizontal bands = drought years (like 2011). Look for the overall "texture" — Texas rivers tend to show scattered, irregular spots rather than the smooth seasonal banding of snowmelt rivers.

---
## 14. Monthly Box Plots

In [ ]:
def plot_monthly_box(obs, title, ax, color=CYAN):
    years = sorted(obs.keys())
    data = [np.array([obs[y][m] for y in years]) for m in range(12)]
    bp = ax.boxplot(data, positions=range(1, 13), widths=0.6, patch_artist=True,
                    showfliers=True, flierprops=dict(marker="o", markersize=3,
                    markerfacecolor=RED, markeredgecolor=RED, alpha=0.5))
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.35)
    for element in ("whiskers", "caps", "medians"):
        for line in bp[element]:
            line.set_color(LIGHT)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, color) in zip(axes, RIVERS):
    plot_monthly_box(obs, f"{name} — Monthly Distribution", ax, color)
plt.tight_layout(); plt.show()

**What this shows:** The statistical distribution of flow for each calendar month across all years of record. The box spans the 25th–75th percentile (interquartile range), the white line inside is the median, whiskers extend to 1.5× IQR, and red dots are outliers beyond that.

**How to read it:** Tall boxes = high variability for that month. Many outlier dots above the whiskers = occasional extreme floods. Months where the median sits near the bottom of the box have right-skewed distributions (a few very wet years pull the mean up). Compare box heights across months to see which season is most predictable vs. most volatile.

---
## 15. Rolling 30-Year Normals

In [ ]:
def plot_rolling_normals(obs, title, ax, color=CYAN):
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    normals = fm.rolling_normals(annual, window=30)
    if normals:
        centers = [years[0] + (n.start + n.end) // 2 for n in normals]
        values = [n.mean for n in normals]
        ax.plot(centers, values, color=color, lw=2.5, label="30-yr rolling normal")
        ax.fill_between(centers, values, alpha=0.15, color=color)
    ax.plot(years, annual, color=MUTED, lw=0.6, alpha=0.5, label="annual")
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=7, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, color) in zip(axes, RIVERS):
    plot_rolling_normals(obs, f"{name} — Rolling Normals", ax, color)
plt.tight_layout(); plt.show()

**What this shows:** The 30-year rolling average of annual flow (bold colored line) overlaid on the raw annual values (faint gray). Climate scientists use 30-year "normals" as the benchmark for a location's typical climate — this shows how that benchmark itself has shifted over the record.

**How to read it:** When the rolling normal rises, the region is in a multi-decadal wet phase; when it dips, a dry phase. The shaded area under the curve makes the trend visually obvious. Compare across rivers: if all three normals rise and fall together, it's a regional climate signal. If they diverge, basin-specific factors (urbanization, reservoir management) are at play.

---
## 16. Cumulative Departure

In [ ]:
def plot_cumulative(obs, title, ax, color=CYAN):
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    mean = np.mean(annual)
    cumulative = np.cumsum(annual - mean)
    ax.plot(years, cumulative, color=color, lw=2)
    ax.fill_between(years, 0, cumulative, alpha=0.15, color=color)
    ax.axhline(0, color=MUTED, lw=0.8, ls=":")
    ax.set_ylabel("cumulative departure (cfs)")
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, color) in zip(axes, RIVERS):
    plot_cumulative(obs, f"{name} — Cumulative Departure", ax, color)
plt.tight_layout(); plt.show()

**What this shows:** The running total of how much each year's flow was above or below the long-term average. Positive = the river has delivered more water than expected up to that point; negative = a cumulative deficit. This is a "water debt/surplus" accounting.

**How to read it:** Rising slopes = wet period (surplus accumulating). Falling slopes = drought (deficit growing). Peaks mark the end of wet epochs; troughs mark drought bottoms. The deepest trough in the record is the worst cumulative drought on record for that gauge. In Central Texas, the 1950s and 2011 droughts typically show as the deepest troughs.

---
## 17. Record Book (All-Time Highs and Lows)

In [ ]:
def plot_record_book(obs, title, ax):
    rb = fm.record_book(obs, n=5)
    labels, values, years, colors = [], [], [], []
    for e in rb.wettest_years[:3]:
        labels.append(f"Wettest #{e.rank}")
        values.append(e.value); years.append(e.year); colors.append(CYAN)
    for e in rb.driest_summers[:3]:
        labels.append(f"Driest Summer #{e.rank}")
        values.append(e.value); years.append(e.year); colors.append(RED)
    for i, (label, val, yr, color) in enumerate(zip(labels, values, years, colors)):
        ax.barh(i, val, color=color, alpha=0.7, height=0.6)
        ax.text(val + 10, i, f"{yr}  ({val:,.0f} cfs)",
                va="center", fontsize=8, color=LIGHT)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("flow (cfs)")
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, (name, obs, _) in zip(axes, RIVERS):
    plot_record_book(obs, f"{name} — Records", ax)
plt.tight_layout(); plt.show()

**What this shows:** The all-time record holders for each river — the wettest and driest years, highest single-month peak, lowest summer baseflow, and highest/lowest annual mean. Each bar's label shows the year and flow value.

**How to read it:** This is the "hall of fame" for each gauge. If recent years appear frequently, the river's extremes are intensifying. If old years dominate, the modern record may be more regulated. Comparing across rivers: do the same years show up as records for multiple basins? That indicates a region-wide event (e.g., the 2015 Memorial Day floods or the 2011 drought).

---
## 18. Analog Years

In [ ]:
def plot_analog_years(obs, title, ax, *, n=8, color=CYAN):
    years = sorted(obs.keys())
    target = max(years)
    analogs = fm.analog_years(obs, target, n=n)
    target_hydro = obs[target]
    m = range(1, 13)
    ax.plot(m, target_hydro, color=color, lw=3, label=f"{target} (target)")
    cmap = plt.get_cmap("plasma")
    for i, a in enumerate(analogs):
        c = cmap(i / n)
        ax.plot(m, obs[a.year], color=c, lw=1.2, alpha=0.7,
                label=f"{a.year} (sim={a.similarity:.3f})")
    ax.set_xticks(list(m))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=6, frameon=False, ncol=2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, color) in zip(axes, RIVERS):
    plot_analog_years(obs, f"{name} — Analog Years to {max(obs)}", ax, color=color)
plt.tight_layout(); plt.show()

**What this shows:** The most recent complete year (bold) compared to its 8 closest historical "analogs" — years whose monthly flow pattern was most similar (by cosine similarity). This answers: "What past years looked most like this year?"

**How to read it:** If analog years cluster in a particular decade, it suggests recurring climate patterns. The similarity score (0–1) in the legend tells you how close the match is — above 0.95 is very similar, below 0.8 is a loose match. This technique is used in seasonal forecasting: "if this year looks like 1992, what happened next?"

---
## 19. Decadal Shift

In [ ]:
def plot_decadal_shift(obs, title, ax):
    years = sorted(obs.keys())
    mid = years[len(years) // 2]
    early = {y: obs[y] for y in years if y < mid}
    late = {y: obs[y] for y in years if y >= mid}
    if early and late:
        early_mean = np.mean([obs[y] for y in early], axis=0)
        late_mean = np.mean([obs[y] for y in late], axis=0)
        m = range(1, 13)
        ax.plot(m, early_mean, color=COOL_BLUE, lw=2,
                label=f"early ({min(early)}–{max(early)})")
        ax.plot(m, late_mean, color=RED, lw=2,
                label=f"late ({min(late)}–{max(late)})")
        ax.fill_between(m, early_mean, late_mean, alpha=0.1,
                        color=CYAN)
        ax.set_xticks(list(m))
        ax.set_xticklabels(MONTH_ABBR, fontsize=7)
        ax.set_ylabel("flow (cfs)")
        ax.set_title(title)
        ax.legend(fontsize=7, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, _) in zip(axes, RIVERS):
    plot_decadal_shift(obs, f"{name} — Early vs. Late Period", ax)
plt.tight_layout(); plt.show()

**What this shows:** The record split in half — the average monthly hydrograph of the "early" period vs. the "late" period. The shaded area between the two curves highlights where and by how much the seasonal pattern has shifted.

**How to read it:** Where the red (late) curve is above the blue (early) curve, flows have increased in that month over time; where blue is above red, flows have declined. If the curves cross, the seasonal shape has changed — perhaps peak flow shifted to a different month. This is a simple but powerful way to visualize regime change without statistical modeling.

---
## 20. Autocorrelation

In [ ]:
def plot_autocorrelation(obs, title, ax, color=CYAN, *, max_lag=15):
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    mean = np.mean(annual)
    std = np.std(annual)
    n = len(annual)
    acf = []
    for lag in range(max_lag + 1):
        if lag >= n:
            acf.append(0.0)
            continue
        c = np.mean((annual[:n-lag] - mean) * (annual[lag:] - mean)) / (std**2)
        acf.append(c)
    lags = list(range(max_lag + 1))
    ax.bar(lags, acf, color=color, alpha=0.7, width=0.6)
    # 95% confidence bounds
    ci = 1.96 / np.sqrt(n)
    ax.axhline(ci, color=AMBER, ls="--", lw=1, label=f"95% CI (±{ci:.2f})")
    ax.axhline(-ci, color=AMBER, ls="--", lw=1)
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_xlabel("lag (years)")
    ax.set_ylabel("autocorrelation")
    ax.set_title(title)
    ax.legend(fontsize=7, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, obs, color) in zip(axes, RIVERS):
    plot_autocorrelation(obs, f"{name} — Autocorrelation", ax, color)
plt.tight_layout(); plt.show()

**What this shows:** The correlation of annual flow with itself at various time lags (1 year, 2 years, ... up to 15 years). This reveals "memory" in the system — does a wet year tend to be followed by another wet year?

**How to read it:** Lag 0 is always 1.0 (a year correlates perfectly with itself). Bars above the amber dashed 95% confidence interval are statistically significant. Positive autocorrelation at lag 1 means wet years tend to cluster (persistence). Significant bars at 2–5 year lags suggest ENSO-scale cycling. If all bars are within the confidence bands, the river has no significant year-to-year memory — each year is essentially independent.

---
## 21. Climate Correlation Matrix

In [ ]:
# ── climate correlation summary ───────────────────────────────────────────
print(f"{'River':<30} {'ONI→Annual r':>14} {'ONI→Peak r':>14} {'PDO→Annual r':>14} {'PDO→Peak r':>14}")
print("─" * 90)
for name, obs, _ in RIVERS:
    years = sorted(obs.keys())
    ann_metric = {y: float(np.nanmean(obs[y])) for y in years}
    peak_metric = {y: float(np.nanmax(obs[y])) for y in years}
    r_oa = fm.correlate(ann_metric, oni)
    r_op = fm.correlate(peak_metric, oni)
    r_pa = fm.correlate(ann_metric, pdo)
    r_pp = fm.correlate(peak_metric, pdo)
    print(f"{name:<30} {r_oa:>+10.3f}       {r_op:>+7.3f}       {r_pa:>+7.3f}       {r_pp:>+7.3f}")

**What this shows:** Pearson correlation (r) between two climate indices (ONI = ENSO, PDO = Pacific Decadal Oscillation) and two flow metrics (annual mean, annual peak) for each river. The p-value tests whether the correlation is statistically significant (p < 0.05).

**How to read it:** Positive r with ONI means El Niño years bring more water (expected for Texas). Positive r with PDO means warm-phase Pacific decades are wetter. Strong r (|r| > 0.3) with low p means the teleconnection is real and exploitable for forecasting. Weak r with high p means that climate index doesn't predict that metric for that basin. Compare across rivers to see which basins are most climate-sensitive.

---
## 22. Comprehensive Metrics Summary

In [ ]:
# ── comprehensive metrics summary table ───────────────────────────────────
for name, obs, _ in RIVERS:
    years = sorted(obs.keys())
    annual = np.array([float(np.mean(obs[y])) for y in years])
    peaks = np.array([float(np.max(obs[y])) for y in years])
    lows = np.array([float(np.min(obs[y][5:9])) for y in years])
    mean_hydro = np.mean([obs[y] for y in years], axis=0)

    mk_ann = fm.mann_kendall(annual)
    mk_peak = fm.mann_kendall(peaks)
    mk_low = fm.mann_kendall(lows)
    tt = fm.center_of_timing_trend(obs)

    print(f"\n{'═'*60}")
    print(f"  {name}")
    print(f"{'═'*60}")
    print(f"  Record:           {min(years)}–{max(years)} ({len(years)} years)")
    print(f"  Annual mean:      {np.mean(annual):,.1f} cfs (σ = {np.std(annual):,.1f})")
    print(f"  Median annual:    {np.median(annual):,.1f} cfs")
    print(f"  CV:               {np.std(annual)/np.mean(annual):.3f}")
    print(f"  Flashiness (med): {np.median([fm.flashiness(obs[y]) for y in years]):.4f}")
    print(f"  COT (mean):       month {fm.center_of_timing(mean_hydro):.1f}")
    print(f"  Seasonal ratio:   {fm.seasonal_ratio(mean_hydro, wet=(10,11,12,1,2,3,4)):.3f}")
    print(f"  ── Trends ──")
    print(f"  Annual:    {mk_ann.trend:>12} (τ={mk_ann.tau:+.3f}, p={mk_ann.p:.4f})")
    print(f"  Peak:      {mk_peak.trend:>12} (τ={mk_peak.tau:+.3f}, p={mk_peak.p:.4f})")
    print(f"  Low-flow:  {mk_low.trend:>12} (τ={mk_low.tau:+.3f}, p={mk_low.p:.4f})")
    print(f"  Timing:    slope {tt.slope:+.4f} mo/yr (p={tt.p:.4f})")

**What this shows:** A quantitative summary of key hydrologic metrics for each river: central tendency (mean, median), variability (CV, flashiness), timing (center of timing, seasonal ratio), and statistical trends (Mann-Kendall for annual/peak/low flows, timing drift slope).

**How to read it:** CV (coefficient of variation) above 0.6 indicates highly variable flow — typical for Texas. Mann-Kendall p < 0.05 means a statistically significant trend. Compare the three rivers: larger basins tend to have more stable flow (lower CV, lower flashiness). Reservoir-regulated reaches may show suppressed peaks and augmented low flows compared to natural basins.

---

## Data Sources & Attribution

All data used in this report are **public domain** or **freely available** from U.S. federal agencies:

| Data | Source | Agency | Access |
|------|--------|--------|--------|
| **Daily mean discharge** (parameter 00060, statistic 00003) | USGS National Water Information System (NWIS) | U.S. Geological Survey | https://waterservices.usgs.gov/nwis/dv/ |
| **ENSO — Oceanic Niño Index (ONI)** | 3-month running mean SST anomaly, Niño 3.4 region | NOAA Climate Prediction Center (CPC) | https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt |
| **Pacific Decadal Oscillation (PDO)** | ERSST v5 monthly PDO index | NOAA National Centers for Environmental Information (NCEI) | https://www.ncei.noaa.gov/pub/data/cmb/ersst/v5/index/ersst.v5.pdo.dat |

### USGS Gauge Stations Used

| Station ID | Station Name | Period Used |
|------------|-------------|-------------|
| 08158000 | Colorado River at Austin, TX | 1960–2024 |
| 08096500 | Brazos River at Waco, TX | 1960–2024 |
| 08176500 | Guadalupe River at Victoria, TX | 1940–2024 |

### Statistical Methods

- **Mann-Kendall test** — non-parametric trend test; does not assume normality. Reports Kendall's τ (strength) and p-value (significance at α = 0.05).
- **Sen's slope** — robust linear trend estimator (median of all pairwise slopes); resistant to outliers.
- **Center of Timing (COT)** — flow-weighted mean month; the calendar date by which 50% of annual streamflow has occurred.
- **Flashiness Index** — Richards-Baker index: sum of absolute month-to-month changes divided by total flow.
- **Seasonal Ratio** — fraction of annual flow arriving in the wet season (Oct–Apr for Central Texas).
- **Analog Years** — cosine similarity between each year's 12-month flow vector and the target year.
- **Flow Duration Curves** — empirical quantile functions of monthly flows, stratified by decade.
- **Composite Hydrographs** — mean monthly flow averaged over years classified as warm (ONI > +0.5), cool (ONI < −0.5), or neutral.

### Reproducibility

All raw NWIS and climate-index data are **snapshotted locally** at fetch time (`/Volumes/home/data/hydro-art/nwis/` and `/Volumes/home/data/hydro-art/climate/`). Subsequent runs parse the local snapshot — no network access required. Flow metrics are computed by `src/flow_metrics.py` (deterministic, offline, numpy-only).

---
*Report generated by hydro-art watershed report toolchain. All source data are U.S. federal public domain.*